# 08 Google Trends download (for the timing model)

Downloads weekly Google Trends interest for the search **term** "dengue" (Brazil + 27 states, 2009–2023) and keeps **all** files on Google Drive in `MyDrive/dengue_recon_data/google_trends/`.

1. Runtime → *Run all*. Allow Google Drive access when asked.
2. Files already in the repository (`data_snapshots/google_trends/`) are copied to Drive first; only the missing ones are downloaded.
3. If Google answers "429" (too many requests), the cell waits and retries. If it stops, run it again later: finished files are kept.
4. When it says `complete: True`, tell Claude: the files on Drive are then committed to GitHub from there.

2024 values are in the last window only to finish epi week 52 of 2023; the analysis drops them before reading.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/dengue-climate-reconstruction")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "timing-model",
                    "https://github.com/aminah9945/dengue-climate-reconstruction", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "pytrends"], check=True)
sys.path.insert(0, str(REPO / "src"))

OUT = Path("/content/drive/MyDrive/dengue_recon_data/google_trends")
OUT.mkdir(parents=True, exist_ok=True)
copied = 0
for f in (REPO / "data_snapshots" / "google_trends").glob("*"):
    if f.is_file() and not (OUT / f.name).exists():
        shutil.copy2(f, OUT / f.name); copied += 1
print("copied from the repository:", copied, "| files on Drive:", len(list(OUT.glob("dengue_*.csv"))))


In [ ]:
from dengue_recon.extensions.timing import trends_download
complete = trends_download.main(["--out", str(OUT), "--sleep", "30"])
print("complete:", complete)


In [ ]:
files = sorted(OUT.glob("dengue_*.csv"))
print(len(files), "files with data on Drive:", OUT)
geos = sorted({f.name.split("_")[1] for f in files})
missing = [f"dengue_{g}_{w[0]}_{w[1]}" for g in trends_download.study_geos() for w in trends_download.WINDOWS
           if not (OUT / f"dengue_{g}_{w[0]}_{w[1]}.csv").exists()]
print("still missing:", missing or "none")
